# YOLO11 DMS 3-class — 12k curated dataset
Kaggle T4 workflow. Outputs `best.pt`, optional `best.onnx`, and `metrics_summary.json`.

In [ ]:
!pip install -q ultralytics pyyaml
import json, shutil, subprocess, sys, zipfile
from pathlib import Path
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

In [ ]:
inputs = Path('/kaggle/input')
dataset_zips = list(inputs.rglob('dms_yolo_3class_v4_12k.zip'))
code_zips = list(inputs.rglob('training_code.zip'))
mounted_yamls = list(inputs.rglob('dms_dataset.yaml'))
mounted_scripts = list(inputs.rglob('train_yolo11_dms.py'))
weights = list(inputs.rglob('yolo11m.pt'))
assert weights, weights
assert dataset_zips or mounted_yamls, (dataset_zips, mounted_yamls)
assert code_zips or mounted_scripts, (code_zips, mounted_scripts)
work = Path('/kaggle/working/dms12k')
data_root, code_root = work/'data', work/'code'
data_root.mkdir(parents=True, exist_ok=True); code_root.mkdir(parents=True, exist_ok=True)
if dataset_zips:
    with zipfile.ZipFile(dataset_zips[0]) as z: z.extractall(data_root)
    dataset_yaml = next(data_root.rglob('dms_dataset.yaml'))
else:
    dataset_yaml = mounted_yamls[0]
if code_zips:
    with zipfile.ZipFile(code_zips[0]) as z: z.extractall(code_root)
    train_script = next(code_root.rglob('train_yolo11_dms.py'))
else:
    train_script = mounted_scripts[0]
print('dataset:', dataset_yaml)
print('trainer:', train_script)
print('weights:', weights[0])

In [ ]:
run_root = Path('/kaggle/working/runs_dms')
run_name = 'yolo11m_dms_3class_v4_12k_768'
cmd = [sys.executable, str(train_script),
       '--data', str(dataset_yaml), '--model', str(weights[0]),
       '--epochs', '60', '--imgsz', '768', '--batch', '8',
       '--device', '0', '--workers', '2', '--patience', '12',
       '--output-dir', str(run_root), '--name', run_name, '--no-onnx']
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
run_dir = run_root/run_name
best_pt = run_dir/'artifacts'/'best.pt'
metrics = run_dir/'artifacts'/'metrics_summary.json'
assert best_pt.exists() and metrics.exists(), (best_pt, metrics)
export = Path('/kaggle/working/dms_export'); export.mkdir(exist_ok=True)
shutil.copy2(best_pt, export/'best.pt'); shutil.copy2(metrics, export/'metrics_summary.json')
try:
    from ultralytics import YOLO
    onnx = Path(str(YOLO(str(best_pt)).export(format='onnx', imgsz=768, simplify=True, dynamic=True)))
    if onnx.exists(): shutil.copy2(onnx, export/'best.onnx')
except Exception as exc:
    print('ONNX export skipped:', exc)
summary = json.loads(metrics.read_text())
print(json.dumps(summary, indent=2)); print('DOWNLOAD:', list(export.iterdir()))